In [2]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 👇 ADD YOUR TRANSFORM HERE
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5], [0.5,0.5,0.5])
])

In [3]:
data_dir = r"C:\Users\Harshu\Downloads\archive (4)\content"

In [4]:
import torch
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
from torchvision import transforms

# Path
data_dir = r"C:\Users\Harshu\Downloads\archive (4)\content"

# Transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Load dataset
dataset = datasets.ImageFolder(data_dir, transform=transform)

# Split
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

generator = torch.Generator().manual_seed(42)

train_data, test_data = torch.utils.data.random_split(
    dataset, [train_size, test_size], generator=generator
)

# Loaders
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

# Debug print
print("Classes:", dataset.classes)
print("Train size:", len(train_data))
print("Test size:", len(test_data))

Classes: ['fake_images', 'real_images']
Train size: 20000
Test size: 5000


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class BetterCNN(nn.Module):
    def __init__(self):
        super(BetterCNN, self).__init__()

        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(128 * 28 * 28, 256)
        self.fc2 = nn.Linear(256, 2)

        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # 224 → 112
        x = self.pool(F.relu(self.conv2(x)))   # 112 → 56
        x = self.pool(F.relu(self.conv3(x)))   # 56 → 28

        x = x.view(-1, 128 * 28 * 28)

        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)

        return x

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BetterCNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
print(device)

cpu


In [11]:
import torch
print(torch.version.cuda)  # which CUDA version PyTorch was built with
print(torch.backends.cudnn.version())  # cuDNN version

None
None


In [14]:
import torch
print(torch.cuda.is_available())  # should be True
print(torch.cuda.get_device_name(0))  # should print RTX 5060

False


AssertionError: Torch not compiled with CUDA enabled

In [8]:
epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}")

Epoch 1, Loss: 0.6743633548736572
Epoch 2, Loss: 0.6520142808914184
Epoch 3, Loss: 0.6164800946712494
Epoch 4, Loss: 0.5737939171791077
Epoch 5, Loss: 0.5240864469051361
Epoch 6, Loss: 0.4762566486597061
Epoch 7, Loss: 0.41376925110816953
Epoch 8, Loss: 0.3457147327184677
Epoch 9, Loss: 0.27207414095401766
Epoch 10, Loss: 0.2080578512787819


In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Accuracy: {100 * correct / total}%")

Accuracy: 70.3%


In [ ]:
torch.save(model.state_dict(), "detector.pt")

NameError: name 'torch' is not defined

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
])